# 파이썬 async/await — 기다리는 시간을 낭비하지 않는다
## 비동기 프로그래밍 완전 정복

> **이 노트북의 목표**
> - 일반 함수와 async 함수의 근본적 차이 이해하기
> - 코루틴(coroutine)이 무엇인지 직접 확인하기
> - `await`, `gather`, `create_task` 각각 언제 쓰는지 체득하기
> - async가 유리한 경우 vs 아닌 경우 판단할 수 있게 되기

---

## Part 0. 먼저 확인 — 함수 정의와 실행의 차이

async를 배우기 전에 이걸 먼저 확인하자.
함수를 `def`로 만드는 것과 `()`로 실행하는 건 완전히 다른 일이다.

In [ ]:
def hello():
    print("안녕")

# 함수 자체를 담기 (실행 X)
a = hello
print(type(a))  # <class 'function'>

# 이때 실행됨
a()

> **핵심 구분**
> - `hello` → 함수 자체 (실행 X)
> - `hello()` → 실행

이 구분이 async에서 훨씬 중요해진다.

---

## Part 1. async def — 왜 호출해도 실행이 안 될까

In [ ]:
# 일반 함수는 호출하면 바로 실행
def normal():
    print("일반 함수 실행")

normal()  # → "일반 함수 실행" 출력

In [ ]:
# async 함수는 호출해도 아무것도 출력 안 됨
async def async_work():
    print("비동기 작업 실행")

result = async_work()  # 실행 안 됨!
print(type(result))    # <class 'coroutine'>
print(result)          # <coroutine object async_work at 0x...>

**무슨 일이 일어났나?**

`async_work()`를 호출하면 실행이 되는 게 아니라 **코루틴 객체**가 만들어진다.

코루틴은:
- "나중에 실행할 비동기 작업표" 같은 것
- `await`를 해줘야 실제로 진행되는 비동기 작업 객체

```
일반 함수 호출  → 바로 실행
async 함수 호출 → 코루틴 객체 생성 (아직 실행 안 됨)
```

In [ ]:
# 코루틴을 실행하지 않고 그냥 두면 경고가 뜸
import warnings
warnings.filterwarnings('error', category=RuntimeWarning)

try:
    coro = async_work()  # 코루틴 생성
    # await 없이 그냥 두면...
    del coro  # garbage collected 시 경고
except RuntimeWarning as e:
    print(f"경고: {e}")

---

## Part 2. await — 코루틴을 실제로 실행시키기

In [ ]:
import asyncio

async def async_work():
    print("비동기 작업 실행")

# await는 async 함수 안에서만 쓸 수 있다
# → 노트북에서는 asyncio.run()으로 실행
asyncio.run(async_work())

In [ ]:
# await의 역할을 직접 확인하기
async def greet(name):
    print(f"{name} 시작")
    await asyncio.sleep(1)   # 1초 비동기 대기
    print(f"{name} 완료")
    return f"{name} 결과"

async def main():
    result = await greet("철수")   # 실행 + 완료 대기
    print(result)

asyncio.run(main())

> **await의 정확한 의미**
> - "이 비동기 작업을 실제로 진행시키고, 끝날 때까지 기다릴게"
> - 단순히 "기다린다"가 아니라 → **코루틴을 실행 흐름에 올리고 + 완료 대기**

---

## Part 3. await를 쓰는 위치 — 어디서나 쓸 수 없다

In [ ]:
# await는 async 함수 안에서만 쓸 수 있다

# 방법 1: async 함수 안에서
async def main():
    await asyncio.sleep(1)   # OK
    result = await greet("영희")  # OK
    print(result)

# 방법 2: 프로그램 시작점 (바깥)에서는 asyncio.run()
asyncio.run(main())

In [ ]:
# 일반 함수 안에서 await를 쓰면?
def not_async():
    # await asyncio.sleep(1)  # SyntaxError 발생!
    pass

# 정리:
# async 함수 안      → await 사용 가능
# 일반 함수 안       → await 사용 불가 (SyntaxError)
# 파일 맨 바깥       → asyncio.run() 사용
print("await는 async 함수 안에서만 쓸 수 있다")

---

## Part 4. time.sleep vs asyncio.sleep — 핵심 차이

In [ ]:
import time

# 동기: 프로그램 전체가 멈춤
start = time.time()
time.sleep(1)  # 1초 동안 아무것도 못 함
print(f"time.sleep: {time.time() - start:.1f}초")

In [ ]:
# 비동기: 나는 쉬는 동안 다른 작업이 진행 가능
async def demo_async_sleep():
    start = time.time()
    await asyncio.sleep(1)  # 비동기 대기
    print(f"asyncio.sleep: {time.time() - start:.1f}초")

asyncio.run(demo_async_sleep())

In [ ]:
# asyncio.sleep()도 async 함수다!
# await 없이 쓰면 코루틴만 만들어지고 실제로 안 쉼

async def wrong_sleep():
    start = time.time()
    asyncio.sleep(1)  # ← await 없음! 코루틴만 생성, 실제 안 쉼
    elapsed = time.time() - start
    print(f"await 없는 sleep: {elapsed:.4f}초 (거의 0초!)")

asyncio.run(wrong_sleep())

| 코드 | 결과 |
|------|------|
| `time.sleep(1)` | 1초 동안 전체 멈춤 |
| `await asyncio.sleep(1)` | 1초 비동기 대기 (다른 작업 가능) |
| `asyncio.sleep(1)` (await 없음) | 아무것도 안 함 (코루틴만 생성) |

---

## Part 5. gather — 여러 작업을 동시에

In [ ]:
# 동기 방식: 순서대로 → 총 6초
async def fetch_sync(name):
    print(f"{name} 요청 시작")
    await asyncio.sleep(2)   # API 대기 시뮬레이션
    print(f"{name} 완료")
    return f"{name} 데이터"

async def main_sync():
    start = time.time()
    await fetch_sync("NVDA")   # 2초 기다림
    await fetch_sync("AMD")    # 또 2초 기다림
    await fetch_sync("TSM")    # 또 2초 기다림
    print(f"순차 실행: {time.time() - start:.1f}초")

asyncio.run(main_sync())

In [ ]:
# gather 방식: 동시에 → 총 2초!
async def main_gather():
    start = time.time()
    results = await asyncio.gather(
        fetch_sync("NVDA"),
        fetch_sync("AMD"),
        fetch_sync("TSM"),
    )
    print(f"gather 실행: {time.time() - start:.1f}초")
    print("결과:", results)

asyncio.run(main_gather())

`asyncio.gather()`의 뜻:

> "이 작업들을 **같이 시작**하고, **전부 끝날 때까지** 기다릴게"

3개가 동시에 진행되므로 6초 → 2초로 줄어든다.

---

## Part 6. create_task — 먼저 시작해두고 나는 다른 일

In [ ]:
# create_task: 작업을 시작해두고 결과를 나중에 받기
async def work(name, delay):
    print(f"{name} 시작")
    await asyncio.sleep(delay)
    print(f"{name} 완료")
    return f"{name} 결과"

async def main_create_task():
    # 작업을 먼저 시작해두기
    task = asyncio.create_task(work("NVDA", 2))

    # 나는 다른 일 할 수 있음
    print("[메인] 다른 처리 중...")
    await asyncio.sleep(0.5)
    print("[메인] 아직 다른 일 하는 중")

    # 나중에 결과 받기
    result = await task
    print(f"[메인] 결과 받음: {result}")

asyncio.run(main_create_task())

In [ ]:
# ⚠️ 흔한 실수: create_task 직후 바로 await하면 장점이 안 보임
async def bad_example():
    task = asyncio.create_task(work("NVDA", 1))
    result = await task   # 바로 기다리면 그냥 await work()와 차이 없음
    print(result)

asyncio.run(bad_example())

| | gather | create_task |
|---|---|---|
| **실행** | 여러 개 같이 시작 | 하나씩 먼저 시작 |
| **기다림** | 전부 끝날 때까지 | 내가 원할 때 await |
| **결과** | 한 번에 리스트로 | await task로 개별 수신 |
| **용도** | 여러 작업 묶어서 처리 | 시작 후 다른 일 가능 |

> **중요**: `create_task`는 결과를 버리는 게 아니다.
> "지금 당장 안 받을 뿐, 나중에 `await task`로 받을 수 있다."

---

## Part 7. gather vs create_task — 직접 비교

In [ ]:
# gather: 여러 개 한 번에
async def demo_gather():
    start = time.time()
    results = await asyncio.gather(
        work("A", 1),
        work("B", 1),
        work("C", 1),
    )
    print(f"gather 총 시간: {time.time() - start:.1f}초")
    print("결과:", results)

asyncio.run(demo_gather())

In [ ]:
# create_task 여러 개: 각각 시작하고 나중에 gather로 기다리기
async def demo_create_tasks():
    start = time.time()

    # 각각 먼저 시작
    t1 = asyncio.create_task(work("X", 1))
    t2 = asyncio.create_task(work("Y", 1))
    t3 = asyncio.create_task(work("Z", 1))

    # 전부 끝날 때까지 기다리기
    results = await asyncio.gather(t1, t2, t3)
    print(f"create_task 총 시간: {time.time() - start:.1f}초")
    print("결과:", results)

asyncio.run(demo_create_tasks())

---

## Part 8. async가 유리한 경우 vs 아닌 경우

In [ ]:
# async가 유리한 경우: I/O 대기 작업
# API 요청, 파일 읽기, DB 대기, 네트워크 통신

async def io_bound_task(name):
    await asyncio.sleep(1)   # I/O 대기 시뮬레이션
    return f"{name} 데이터"

async def main_io():
    start = time.time()
    results = await asyncio.gather(
        io_bound_task("API_1"),
        io_bound_task("API_2"),
        io_bound_task("API_3"),
        io_bound_task("API_4"),
        io_bound_task("API_5"),
    )
    print(f"5개 동시 처리: {time.time() - start:.1f}초 (1초!)")

asyncio.run(main_io())

In [ ]:
# async가 의미 없는 경우: CPU 계산 작업
import math

async def cpu_bound_task(n):
    # 순수 계산 — await가 없으면 다른 작업과 번갈아가며 실행 불가
    result = sum(math.sqrt(i) for i in range(n))
    return result

async def main_cpu():
    start = time.time()
    results = await asyncio.gather(
        cpu_bound_task(1_000_000),
        cpu_bound_task(1_000_000),
    )
    print(f"CPU 계산 2개: {time.time() - start:.2f}초")
    print("→ CPU 작업은 async로 빨라지지 않음")

asyncio.run(main_cpu())

| 유리한 경우 (I/O) | 아닌 경우 (CPU) |
|---|---|
| API 요청 | 복잡한 수치 계산 |
| 웹 크롤링 | 반복문으로 CPU 많이 쓰는 일 |
| 파일 읽기 | 숫자 연산 집약적 작업 |
| DB 대기 | |
| 네트워크 통신 | |

> async의 핵심: **기다리는 시간(I/O)을 다른 작업으로 채우는 것**
> CPU 계산에는 쉬는 시간이 없으므로 async가 의미 없다.

---

## Part 9. asyncio 탐색하는 법 — 실전 팁

In [ ]:
# asyncio 안에 뭐가 있나 보기
import asyncio
[x for x in dir(asyncio) if not x.startswith('_')]

In [ ]:
# 특정 함수 설명 보기
help(asyncio.gather)

In [ ]:
# 실제 소스 보기
import inspect
# inspect.getsource(asyncio.sleep)  # 실제 구현 코드 확인

---

## 최종 확인 문제

실행 전에 결과를 예측해보자.

In [ ]:
# Q1. 이 코드의 출력 결과는?
async def test_q1():
    print("시작")

result = test_q1()  # await 없음
print(type(result))
# 예측: ?

In [ ]:
# Q2. 총 실행 시간은?
async def q2_task():
    await asyncio.sleep(1)

async def q2_main():
    start = time.time()
    await asyncio.gather(
        q2_task(),
        q2_task(),
        q2_task(),
    )
    print(f"총 시간: {time.time() - start:.1f}초")
    # 예측: 1초? 3초?

asyncio.run(q2_main())

In [ ]:
# Q3. await 없이 asyncio.sleep을 쓰면?
async def q3_main():
    start = time.time()
    asyncio.sleep(2)   # await 없음!
    elapsed = time.time() - start
    print(f"경과 시간: {elapsed:.3f}초")
    # 예측: 2초? 0초?

asyncio.run(q3_main())

In [ ]:
# Q4. create_task vs 직접 await — 시간 차이가 있나?
async def slow_task(name):
    await asyncio.sleep(1)
    return name

async def q4_main():
    start = time.time()

    t1 = asyncio.create_task(slow_task("A"))
    t2 = asyncio.create_task(slow_task("B"))
    t3 = asyncio.create_task(slow_task("C"))

    r1 = await t1
    r2 = await t2
    r3 = await t3

    print(f"결과: {r1}, {r2}, {r3}")
    print(f"시간: {time.time() - start:.1f}초")
    # 예측: 1초? 3초?

asyncio.run(q4_main())

---

## 핵심 요약

| 개념 | 핵심 |
|------|------|
| `async def work()` | 비동기 함수 정의 |
| `work()` | 코루틴 객체 생성 (실행 아님) |
| `await work()` | 코루틴 실제 실행 + 완료 대기 |
| `await`는 async 안에서만 | 바깥에서는 `asyncio.run()` |
| `asyncio.sleep()`도 async | 반드시 `await` 붙여야 함 |
| `gather` | 여러 작업 동시 실행 + 전체 대기 |
| `create_task` | 작업 먼저 시작, 결과는 나중에 |
| async는 I/O에 강함 | CPU 계산에는 의미 없음 |

> **한 줄 정리**
> async의 핵심은 "기다리는 시간을 낭비하지 않는 것"이다.
> `await`는 "지금 기다리는 동안 다른 작업이 진행될 수 있게 허용하는 신호"다.